In [ ]:
# Submission path setup: run notebooks from any submission subfolder.
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'Functions.ipynb').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)


In [ ]:
from pathlib import Path
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)
pd.set_option("display.max_rows", 200)


In [ ]:
base_csv = Path("early_search/classification_segmentation_benchmark_kfold.csv")
extended_csv = Path("early_search/classification_segmentation_benchmark_kfold_big.csv")

base_df = pd.read_csv(base_csv)
extended_df = pd.read_csv(extended_csv)

print(f"{base_csv.name}: rows={len(base_df)}, cols={len(base_df.columns)}")
print(f"{extended_csv.name}: rows={len(extended_df)}, cols={len(extended_df.columns)}")


In [ ]:
print(base_csv.name)
display(base_df)

print(extended_csv.name)
display(extended_df)


In [ ]:
extended_compare_df = extended_df.copy()
extended_compare_df["base_run_name"] = extended_compare_df["run_name"].str.replace(r"_lenient_ep\d+_min\d+_pat\d+$", "", regex=True)

common_cols = [col for col in base_df.columns if col in extended_compare_df.columns and col != "run_name"]

for _, row in extended_compare_df.iterrows():
    pair_df = pd.concat([
        base_df.loc[base_df["run_name"] == row["base_run_name"], ["run_name"] + common_cols].assign(version="before"),
        extended_compare_df.loc[extended_compare_df["run_name"] == row["run_name"], ["run_name"] + common_cols].assign(version="after"),
    ], ignore_index=True)

    print(f"Base run: {row['base_run_name']}")
    display(pair_df[["version", "run_name"] + common_cols])


In [ ]:
plot_df = extended_df.copy().reset_index(drop=True)
plot_df["model_id"] = [f"M{i+1}" for i in range(len(plot_df))]

split_rows = [
    ("train", "Train", "#4C72B0"),
    ("val", "Validation", "#55A868"),
    ("test", "Test", "#C44E52"),
]

metric_cols = [
    ("dice", "Dice"),
    ("hd95", "HD95 Distance"),
    ("macro_f1", "Macro F1"),
]

def resolve_metric_col(metric_key, split_key):
    plain = f"{metric_key}_{split_key}"
    if plain in plot_df.columns:
        return plain
    match_cols = [c for c in plot_df.columns if c.startswith(f"{metric_key}_{split_key} ")]
    if len(match_cols) != 1:
        raise KeyError(f"Could not uniquely find column for {plain}")
    return match_cols[0]

def padded_ylim(values, metric_key=None):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    lo = float(vals.min())
    hi = float(vals.max())
    span = hi - lo
    pad = 0.08 * span if span > 0 else 0.03
    if metric_key in ["dice", "macro_f1"]:
        lo = max(0.0, lo - pad)
        hi = min(1.0, hi + pad)
        if hi - lo < 0.10:
            mid = (hi + lo) / 2
            lo = max(0.0, mid - 0.05)
            hi = min(1.0, mid + 0.05)
    else:
        lo = max(0.0, lo - pad)
        hi = hi + pad
        if hi - lo < 0.10:
            hi = lo + 0.10
    return lo, hi

fig, axes = plt.subplots(4, 3, figsize=(18, 17))
x = np.arange(len(plot_df))

for row_idx, (split_key, split_label, color) in enumerate(split_rows):
    for col_idx, (metric_key, metric_label) in enumerate(metric_cols):
        ax = axes[row_idx, col_idx]
        col_name = resolve_metric_col(metric_key, split_key)
        vals = plot_df[col_name].to_numpy()

        ax.bar(x, vals, color=color, alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(plot_df["model_id"])
        ax.set_ylim(*padded_ylim(vals, metric_key=metric_key))
        ax.set_title(f"{split_label}: {metric_label}")
        if row_idx == 3:
            ax.set_xlabel("Extended Model")
        if col_idx == 0:
            ax.set_ylabel("Score")

for col_idx, (metric_key, metric_label) in enumerate(metric_cols):
    ax = axes[3, col_idx]
    width = 0.24
    offsets = [-width, 0.0, width]
    combined_vals = []

    for offset, (split_key, split_label, color) in zip(offsets, split_rows):
        col_name = resolve_metric_col(metric_key, split_key)
        vals = plot_df[col_name].to_numpy()
        combined_vals.extend(vals.tolist())
        ax.bar(x + offset, vals, width=width, color=color, alpha=0.85, label=split_label)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["model_id"])
    ax.set_ylim(*padded_ylim(combined_vals, metric_key=metric_key))
    ax.set_title(f"Combined: {metric_label}")
    ax.set_xlabel("Extended Model")
    if col_idx == 0:
        ax.set_ylabel("Score")

handles, labels = axes[3, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.985), ncol=3, frameon=False)
fig.suptitle("Extended models: split-by-metric comparison", y=0.998, fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.965])
plt.show()

display_cols = [
    "model_id", "run_name",
    "dice_train", "dice_val", "dice_test",
    "hd95_train", "hd95_val", "hd95_test",
    "macro_f1_train (<0.990)", "macro_f1_val (<0.907)", "macro_f1_test (<0.840)",
]
display(plot_df[display_cols])


In [ ]:
plot_df = extended_df.copy().reset_index(drop=True)
plot_df["model_id"] = [f"M{i+1}" for i in range(len(plot_df))]

split_rows = [
    ("train", "Train", "#4C72B0"),
    ("val", "Validation", "#55A868"),
    ("test", "Test", "#C44E52"),
]

metric_cols = [
    ("dice", "Dice"),
    ("hd95", "HD95 Distance"),
    ("macro_f1", "Macro F1"),
]

def resolve_metric_col(metric_key, split_key):
    plain = f"{metric_key}_{split_key}"
    if plain in plot_df.columns:
        return plain
    match_cols = [c for c in plot_df.columns if c.startswith(f"{metric_key}_{split_key} ")]
    if len(match_cols) != 1:
        raise KeyError(f"Could not uniquely find column for {plain}")
    return match_cols[0]

def padded_ylim(values, metric_key=None):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    lo = float(vals.min())
    hi = float(vals.max())
    span = hi - lo
    pad = 0.08 * span if span > 0 else 0.03
    if metric_key in ["dice", "macro_f1"]:
        lo = max(0.0, lo - pad)
        hi = min(1.0, hi + pad)
        if hi - lo < 0.10:
            mid = (hi + lo) / 2
            lo = max(0.0, mid - 0.05)
            hi = min(1.0, mid + 0.05)
    else:
        lo = max(0.0, lo - pad)
        hi = hi + pad
        if hi - lo < 0.10:
            hi = lo + 0.10
    return lo, hi

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
x = np.arange(len(plot_df))

for ax, (metric_key, metric_label) in zip(axes, metric_cols):
    width = 0.24
    offsets = [-width, 0.0, width]
    combined_vals = []

    for offset, (split_key, split_label, color) in zip(offsets, split_rows):
        col_name = resolve_metric_col(metric_key, split_key)
        vals = plot_df[col_name].to_numpy()
        combined_vals.extend(vals.tolist())
        ax.bar(x + offset, vals, width=width, color=color, alpha=0.85, label=split_label)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["model_id"])
    ax.set_ylim(*padded_ylim(combined_vals, metric_key=metric_key))
    ax.set_title(metric_label)
    ax.set_xlabel("Segmentation Model")

axes[0].set_ylabel("Score")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.05), ncol=3, frameon=False)
fig.suptitle("Segmentation Models: train / validation / test set", y=1.12, fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()
